### 1. Configuración Inicial e Instalación de Librerías
Comenzamos instalando y cargando las librerías necesarias para el análisis de datos, el modelado y la interfaz interactiva con Gradio.

In [ ]:
!pip install -q gradio tensorflow scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import gradio as gr
import os

# Configuración de estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 2. Carga del Dataset y Análisis Exploratorio de Datos (EDA)
Cargamos el set de datos de California Housing y describimos sus columnas.

In [ ]:
# Cargar dataset de California Housing
california = fetch_california_housing(as_frame=True)
df = california.frame

# Descripción de las columnas:
# - MedInc: Ingreso medio de los hogares en un bloque (en decenas de miles de dólares).
# - HouseAge: Edad mediana de las casas en un bloque.
# - AveRooms: Promedio de habitaciones por hogar.
# - AveBedrms: Promedio de dormitorios por hogar.
# - Population: Población total del bloque.
# - AveOccup: Promedio de ocupantes por hogar.
# - Latitude: Latitud geográfica del bloque.
# - Longitude: Longitud geográfica del bloque.
# - MedHouseValue: Valor mediano de la vivienda para los bloques de California (en cientos de miles de dólares) - VARIABLE OBJETIVO.

print("Primeras filas del dataset:")
display(df.head())
print("\nEstadísticas descriptivas:")
display(df.describe())

Primeras filas del dataset:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422



Estadísticas descriptivas:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


### 3. Entrenamiento de la Red Neuronal para Predicción de Precios
Preparamos los datos escalándolos y creamos un modelo de Red Neuronal Regresora mediante TensorFlow.

In [ ]:
# División en características (X) y objetivo (y)
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Partición de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalado de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Creación de la arquitectura de la Red Neuronal
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.1),
    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dense(1)  # Salida única para regresión
])

# Compilación de la red neuronal
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Entrenamiento del modelo
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=40,
    batch_size=64,
    verbose=1
)

### 4. Evaluación del Modelo y Métricas de Regresión
Calculamos las métricas requeridas: MSE, RMSE, MAE, MAPE, R2 y graficamos el histograma de errores junto con un ejemplo de predicción individual.

In [ ]:
# Asegurarse de que el modelo y las variables están listos, predecir sobre el set de prueba
y_pred = model.predict(X_test_scaled).flatten()

# Cálculo de métricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
r2 = r2_score(y_test, y_pred)

print(f"--- Métricas de Regresión ---")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Coeficiente de Determinación (R2): {r2:.4f}\n")

# Graficar histograma de errores (residuos)
errores = y_test - y_pred
plt.figure(figsize=(10, 5))
sns.histplot(errores, kde=True, bins=50, color='purple')
plt.title("Distribución de los Errores de Predicción (Residuos)")
plt.xlabel("Error (Valor Real - Valor Predicho)")
plt.ylabel("Frecuencia")
plt.savefig('histograma_errores.png')
plt.show()

# Ejemplo de predicción individual
ejemplo_idx = 0
ejemplo_x = X_test.iloc[[ejemplo_idx]]
ejemplo_x_scaled = X_test_scaled[[ejemplo_idx]]
prediccion_ejemplo = model.predict(ejemplo_x_scaled)[0][0]
valor_real = y_test.iloc[ejemplo_idx]

print(f"--- Ejemplo de Predicción ---")
print(f"Datos de entrada de la casa (Valores reales):\n{ejemplo_x.to_string(index=False)}")
print(f"Valor Real de la casa: ${valor_real * 100000:.2f} USD")
print(f"Valor Predicho por la Red Neuronal: ${prediccion_ejemplo * 100000:.2f} USD")

### 5. Generación de Gráficos para la Interfaz de Gradio
Generamos y guardamos las imágenes correspondientes a los análisis univariados, bivariados, multivariados y la matriz de correlación para que Gradio las visualice estáticamente en su respectiva pestaña.

In [ ]:
# 1. Distribución Univariada (Precio de la vivienda)
plt.figure(figsize=(8, 4))
sns.histplot(df['MedHouseVal'], kde=True, color='blue')
plt.title('Distribución del Valor Medio de la Vivienda (MedHouseVal)')
plt.xlabel('Valor en Cientos de Miles de USD')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('dist_univariada.png')
plt.close()

# 2. Distribución Bivariada (Precio vs Ingreso Medio)
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df.sample(1000, random_state=42), x='MedInc', y='MedHouseVal', alpha=0.5, color='orange')
plt.title('Relación: Ingreso Medio vs Valor de la Vivienda')
plt.xlabel('Ingreso Medio (MedInc)')
plt.ylabel('Valor de la Vivienda (MedHouseVal)')
plt.tight_layout()
plt.savefig('dist_bivariada.png')
plt.close()

# 3. Distribución Multivariada (Mapa Geográfico basado en Coordenadas y Precio)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(df['Longitude'], df['Latitude'], c=df['MedHouseVal'], cmap='coolwarm', alpha=0.3, s=df['Population']/100)
plt.colorbar(scatter, label='Valor Medio de la Vivienda')
plt.title('Distribución Multivariada: Ubicación vs Población y Precio')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.tight_layout()
plt.savefig('dist_multivariada.png')
plt.close()

# 4. Matriz de Correlación
plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlación de California Housing')
plt.tight_layout()
plt.savefig('matriz_correlacion.png')
plt.close()

### 6. Construcción y Lanzamiento del Dashboard de Gradio
Organizamos toda la información en pestañas integradas dentro de un solo bloque interactivo para poder explorarlo en una sola pantalla.

In [ ]:
import gradio as gr

# Contenidos teóricos en Markdown
explicacion_ejercicio = """
# Explicación del Ejercicio de Predicción de Vivienda en California

Este proyecto implementa una solución de Machine Learning integral utilizando un set de datos clásico: **California Housing**.

### Pasos del Proceso:
1. **Ingesta de Datos**: Lectura y formateo de la información con pandas.
2. **EDA (Análisis Exploratorio)**: Comprensión de variables mediante estadísticas descriptivas, histogramas de distribución, análisis de relaciones y mapas de calor correlacionales.
3. **Preprocesamiento**: Tratamiento del dataset dividiéndolo en conjunto de entrenamiento/test y aplicando un StandardScaler para optimizar la convergencia de la red neuronal.
4. **Entrenamiento de Red Neuronal**: Arquitectura multicapa (MLP) en Keras usando capas ocultas, Dropout para mitigar el sobreajuste (overfitting) y optimizador Adam.
5. **Evaluación de Métricas**: Comparativa matemática de los resultados de regresión y análisis de residuos mediante el histograma de errores.
"""

conclusiones_ejercicio = """
# Conclusiones Generales del Ejercicio

- **La variable más determinante**: La correlación y los gráficos univariados confirman que el **Ingreso Medio (MedInc)** es la variable que más peso tiene sobre el precio final de las casas.
- **Ubicación Geográfica**: El análisis geográfico (Latitud/Longitud) resalta que los precios se elevan drásticamente en zonas costeras específicas (San Francisco, Los Ángeles), demostrando que la ubicación influye altamente de manera no-lineal.
- **Desempeño de la Red Neuronal**: Las métricas muestran un error absoluto medio (MAE) bajo en relación al rango total del precio del dataset. Un error porcentual promedio (MAPE) controlado y un R2 alto confirman la idoneidad del modelado secuencial de deep learning en lugar de regresores lineales básicos.
- **Distribución del Error**: El histograma de residuos sigue una distribución aproximadamente gaussiana centrada en cero, lo que verifica que las predicciones del modelo son estables y no están sesgadas sistemáticamente.
"""

# Crear la interfaz de Gradio con Pestañas (Tabs)
with gr.Blocks() as demo:
    gr.Markdown("## 🏠 California Housing - Dashboard de Análisis y Predicción (Red Neuronal)")

    with gr.Tab("Vista del DataFrame"):
        gr.Markdown("### Primeras Filas del Dataset")
        gr.DataFrame(df.head(50))

    with gr.Tab("Estadísticas"):
        gr.Markdown("### Estadísticas Descriptivas Generales")
        gr.DataFrame(df.describe())

    with gr.Tab("Distribuciones Univariadas"):
        gr.Markdown("### Distribución de la Variable Objetivo")
        gr.Image("dist_univariada.png")

    with gr.Tab("Análisis Bivariado"):
        gr.Markdown("### Relación entre el Ingreso Medio y el Valor de la Vivienda")
        gr.Image("dist_bivariada.png")

    with gr.Tab("Análisis Multivariado"):
        gr.Markdown("### Geografía de California correlacionada con los precios de viviendas")
        gr.Image("dist_multivariada.png")

    with gr.Tab("Matriz de Correlación"):
        gr.Markdown("### Mapa de Calor de Correlación de Pearson")
        gr.Image("matriz_correlacion.png")

    with gr.Tab("Métricas de la Red Neuronal"):
        gr.Markdown(f"""
        ### Desempeño del Modelo de Red Neuronal
        - **Mean Squared Error (MSE)**: {mse:.4f}
        - **Root Mean Squared Error (RMSE)**: {rmse:.4f}
        - **Mean Absolute Error (MAE)**: {mae:.4f}
        - **Mean Absolute Percentage Error (MAPE)**: {mape:.2f}%
        - **R² Score**: {r2:.4f}
        """)
        gr.Image("histograma_errores.png")

    with gr.Tab("Explicación del Ejercicio"):
        gr.Markdown(explicacion_ejercicio)

    with gr.Tab("Conclusiones"):
        gr.Markdown(conclusiones_ejercicio)

# Lanzar la aplicación web de Gradio
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64d1c79fad1c8699f9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
